# 01. Indexing

**Tier:** Core. **Why you'd use this:** A receipt line is one string. You need the first character, a middle slice for the SKU, and a safe slice that does not crash when the line is short.

**After this lesson you can:** take a character or a slice, including one that runs past the end; explain why a partial slice is a new string; read an `IndexError` when an index is out of range.
About 20 minutes. **Uses:** [Names](../01-basics/01-names-lesson.ipynb)


## Quick review

1. How do you test for a missing value with `None`?
2. What does `else` on an `if` chain catch?

<details><summary>Answers</summary>

`value is None`. `else` runs when every branch above was false.
</details>


## The idea

A string is a row of code points. Indexes start at 0, and `-1` is the last. A slice keeps the start and leaves out the stop. `s[::-1]` reverses the text.

<details><summary>Explain more</summary>

`len` counts code points. The last valid index is `len(s) - 1`. One past that raises `IndexError`. A slice past the end stops instead.
</details>


## See it work

Hide the middle of a card number, leaving the first four and last four characters visible.


In [1]:
def mask_card(number):
    # 1. Keep the first four characters.
    start = number[:4]
    # 2. Keep the last four.
    end = number[-4:]
    # 3. The middle is whatever those two slices did not take.
    hidden = "*" * (len(number) - 8)
    return start + hidden + end


print(mask_card("4111111111111111"))
print(mask_card("12345678"))


4111********1111
12345678


## Check yourself

Answer from the example before you run.


1. Why do `word[:2]` and `word[2:]` rebuild `hello`?
2. Why is `len("é")` smaller than the UTF-8 length?

<details><summary>Answers</summary>

The second slice starts where the first stopped. `len` counts code points; UTF-8 may use more bytes.
</details>


In [2]:
word = "hello"
print(word[0], word[-1], word[1:4])
print(word[::-1])
print(repr(word[-2:0]))
print(repr(word[0:100]))
encoded = "é".encode()
print(len("é"), len(encoded), encoded)


h o ell
olleh
''
'hello'
1 2 b'\xc3\xa9'


<details><summary>Check your answers</summary>

- `word[1:4]` is `ell`.
- `word[::-1]` is `olleh`.
- `word[-2:0]` is empty — step walks forward, and 0 is behind the start.
- `word[0:100]` is the whole word.
- `len("é")` is 1; UTF-8 encode is 2 bytes.
</details>


## Why it works this way

A string stores Unicode code points, one slot each. `é` is one code point (`len` is 1) and two UTF-8 bytes. Slices count slots, so they can split a letter that looks single.

The stop is excluded so `word[:i]` and `word[i:]` rebuild the original. A stop past the end does not raise; an index past the end does.

A partial slice copies into a new string. A full slice can be the same object — CPython detail: a copy would waste memory. Assigning to `word[0]` fails: fixed slots keep the hash valid for dict keys. Deep dive: [Numbers and strings](../../internals/05-numbers/01-numbers-strings-lesson.ipynb). Indexing a slot is O(1); keeping many partial slices still costs memory.


In [3]:
word = "hello"
print(word[:2] + word[2:] == word)
print(word is word[0:4])
print(word is word[:])
print(len("e\u0301"))


True
False
True
2


In [4]:
word = "hello"
# Slots are fixed. This tries to write into the object.
word[0] = "H"


TypeError: 'str' object does not support item assignment

## When to use it

Use an index for one code point and a slice for a span. Use `-1` for the last character. To change text, build a new string.


## Common mistake

`word[-2:0]` looks like the last two characters. It is empty. The stop is at the front, and the default step walks forward, so the slice never moves.


In [5]:
def last_two(word):
    return word[-2:0]


print(repr(last_two("hello")))


''


Omit the stop. That means "through the end," so `-2` can walk forward.


In [6]:
def last_two(word):
    return word[-2:]


print(repr(last_two("hello")))


'lo'


Using `len` as an index raises. Read the last line first: `IndexError`. The arrow marks the bad index. `len(word)` is one past the last slot.


In [7]:
def last_character(word):
    index = len(word)
    return word[index]


print(last_character("hello"))


IndexError: string index out of range

## Try it

Change `mask_card` so it keeps the first 6 characters and the last 4. What does it return for `"12345678"`, and why do those two slices overlap?


## Recap

- Indexes start at 0. A slice leaves out its stop.
- A slice past the end returns what it can; an index past the end raises.
- A partial slice copies.

Next: [practice](01-indexing-practice.ipynb).
